# EDA — ViGoEmotions Ekman-7
**DAP391m · Nhóm 8 · FA26**

Phân tích dữ liệu ViGoEmotions sau khi ánh xạ 28 nhãn GoEmotions xuống 7 nhãn Ekman + Neutral.
Gắn trực tiếp với 3 Research Questions:
- **RQ1**: Các mô hình hoạt động hiệu quả như thế nào sau khi ánh xạ 28 → 7?
- **RQ2**: Những cảm xúc nào mô hình vẫn còn nhầm lẫn? Nhầm lẫn do dữ liệu hay do mô hình?
- **RQ3**: Kỹ thuật nào có thể giảm nhầm lẫn? (Loss function, chuẩn hóa...)


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import json
import ast
import duckdb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR = Path("../ViGoEmotions_Ekman7 (1)/data_ekman7_neutral_wins")

# Kaggle path override
import os
if not DATA_DIR.exists():
    DATA_DIR = Path("/kaggle/input/vigoekman7/data_ekman7_neutral_wins")

print("DATA_DIR:", DATA_DIR)
print("Files:", [f.name for f in DATA_DIR.iterdir()])


## 2. Load Data — số dòng trước và sau tiền xử lý

In [ ]:
# Load 3 splits
train_raw = pd.read_csv(DATA_DIR / "train.csv", keep_default_na=False)
val_raw   = pd.read_csv(DATA_DIR / "val.csv",   keep_default_na=False)
test_raw  = pd.read_csv(DATA_DIR / "test.csv",  keep_default_na=False)

print("=== Số dòng ban đầu ===")
print(f"Train : {len(train_raw):,}")
print(f"Val   : {len(val_raw):,}")
print(f"Test  : {len(test_raw):,}")
print(f"Tổng  : {len(train_raw)+len(val_raw)+len(test_raw):,}")


In [ ]:
# Columns overview
print("Columns:", train_raw.columns.tolist())
train_raw.head(3)


In [ ]:
# Load mapping
mapping = json.load(open(DATA_DIR / "ekman_mapping.json"))
label_dict = json.load(open(DATA_DIR / "label_dict_ekman7.json"))
LABEL_NAMES = [label_dict[str(i)] for i in range(7)]
print("7 Ekman labels:", LABEL_NAMES)
print()
print("Mapping (28 → 7):")
for group, orig_labels in mapping["groups"].items():
    print(f"  {group:10s} ← {orig_labels}")


In [ ]:
# Tiền xử lý: explode label_names thành list
def parse_label_names(df):
    df = df.copy()
    df["label_list"] = df["label_names"].apply(lambda x: str(x).split("|"))
    return df

train = parse_label_names(train_raw)
val   = parse_label_names(val_raw)
test  = parse_label_names(test_raw)

# Conflict resolution: neutral_wins
n_neutral_wins_train = (train_raw["neutral_mixed"] == True).sum()
print(f"Samples có conflict (neutral wins) trong train: {n_neutral_wins_train:,} "
      f"({n_neutral_wins_train/len(train)*100:.1f}%)")

print()
print("=== Số dòng sau tiền xử lý (giữ nguyên — không drop dòng nào) ===")
print(f"Train : {len(train):,}")
print(f"Val   : {len(val):,}")
print(f"Test  : {len(test):,}")


## 3. Ánh xạ 28 → 7 nhãn Ekman (RQ1 context)

Bước tiền xử lý chính của đề tài: gộp 28 nhãn GoEmotions thành 7 nhãn Ekman.
Conflict resolution: nếu sample có cả `neutral` lẫn nhãn khác → giữ `neutral` (*neutral wins*).


In [ ]:
# Số nhãn gốc mỗi nhóm Ekman
group_sizes = {g: len(lbls) for g, lbls in mapping["groups"].items()}
print("Số nhãn gốc mỗi nhóm Ekman:")
for g, n in sorted(group_sizes.items(), key=lambda x: -x[1]):
    print(f"  {g:10s}: {n} nhãn")

fig, ax = plt.subplots(figsize=(8, 3.5))
groups_sorted = sorted(group_sizes.items(), key=lambda x: -x[1])
colors = ["#2a78d6","#eb6834","#1baf7a","#eda100","#e87ba4","#008300","#6250d6"]
ax.bar([g for g,_ in groups_sorted], [n for _,n in groups_sorted],
       color=colors[:len(groups_sorted)], width=0.6)
ax.set_ylabel("Số nhãn gốc")
ax.set_title("Số nhãn GoEmotions gộp vào mỗi nhóm Ekman-7")
ax.set_ylim(0, max(group_sizes.values()) + 2)
for i, (g, n) in enumerate(groups_sorted):
    ax.text(i, n + 0.2, str(n), ha="center", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/figures/mapping_group_sizes.png", dpi=150, bbox_inches="tight")
plt.show()
print("Insight: joy gộp 12 nhãn (nhiều nhất), disgust và neutral giữ nguyên 1 nhãn.")


## 4. Phân phối nhãn (RQ1 & RQ3 context)

In [ ]:
def count_labels(df):
    c = Counter()
    for row in df["label_list"]:
        for lbl in row:
            c[lbl.strip()] += 1
    return pd.Series({lbl: c.get(lbl, 0) for lbl in LABEL_NAMES})

dist_train = count_labels(train)
dist_val   = count_labels(val)
dist_test  = count_labels(test)

dist_df = pd.DataFrame({"train": dist_train, "val": dist_val, "test": dist_test})
print(dist_df)
print()
print(f"Nhãn nhiều nhất (train): {dist_train.idxmax()} — {dist_train.max():,}")
print(f"Nhãn ít nhất  (train): {dist_train.idxmin()} — {dist_train.min():,}")
print(f"Tỉ lệ mất cân bằng: {dist_train.max()/dist_train.min():.1f}x")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(LABEL_NAMES))
w = 0.26
colors_split = ["#2a78d6", "#eb6834", "#1baf7a"]
for i, (split, col) in enumerate(zip(["train","val","test"], colors_split)):
    vals = [dist_df.loc[lbl, split] for lbl in LABEL_NAMES]
    bars = ax.bar(x + (i-1)*w, vals, width=w, label=split, color=col)

ax.set_xticks(x)
ax.set_xticklabels(LABEL_NAMES)
ax.set_ylabel("Sample count")
ax.set_title("Label distribution across splits — Ekman-7")
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
plt.tight_layout()
plt.savefig("../reports/figures/label_distribution_ekman7.png", dpi=150, bbox_inches="tight")
plt.show()
print("Insight: joy chiếm ~43% label trong train. neutral và disgust rất thấp → "
      "mất cân bằng nghiêm trọng, cần pos_weight (RQ3).")


## 5. Thống kê multi-label

In [ ]:
train["n_labels"] = train["label_list"].apply(len)
val["n_labels"]   = val["label_list"].apply(len)
test["n_labels"]  = test["label_list"].apply(len)

for split, df in [("Train", train), ("Val", val), ("Test", test)]:
    print(f"{split}:")
    print(f"  Trung bình nhãn/sample : {df['n_labels'].mean():.3f}")
    print(f"  Sample 1 nhãn          : {(df['n_labels']==1).sum():,} ({(df['n_labels']==1).mean()*100:.1f}%)")
    print(f"  Sample ≥2 nhãn         : {(df['n_labels']>=2).sum():,} ({(df['n_labels']>=2).mean()*100:.1f}%)")
    print()


## 6. Độ dài văn bản — Đơn biến (Histogram + KDE + Boxplot)

In [ ]:
train["text_len"] = train["text"].apply(len)
print("Thống kê độ dài văn bản (ký tự) — train:")
print(train["text_len"].describe().round(1))
print(f"\n75th percentile: {train['text_len'].quantile(0.75):.0f} ký tự")
print(f"max_len=200 bao phủ: {(train['text_len']<=200).mean()*100:.1f}% samples")


In [ ]:
# Histogram + KDE (đơn biến)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

# Histogram + KDE
ax = axes[0]
ax.hist(train["text_len"].clip(upper=300), bins=50, color="#2a78d6",
        alpha=0.7, density=True, label="Histogram")
from scipy.stats import gaussian_kde
kde = gaussian_kde(train["text_len"].clip(upper=300))
xs = np.linspace(0, 300, 300)
ax.plot(xs, kde(xs), color="#eb6834", linewidth=2, label="KDE")
ax.axvline(200, color="#e34948", linestyle="--", linewidth=1.5, label="max_len=200")
ax.set_xlabel("Độ dài văn bản (ký tự)")
ax.set_ylabel("Density")
ax.set_title("Phân phối độ dài văn bản")
ax.legend(fontsize=9)

# Boxplot số nhãn/sample theo emotion (hai biến preview)
ax2 = axes[1]
# Số nhãn per sample
n_label_counts = train["n_labels"].value_counts().sort_index()
ax2.bar(n_label_counts.index.astype(str), n_label_counts.values,
        color="#1baf7a", width=0.5)
ax2.set_xlabel("Số nhãn trên 1 sample")
ax2.set_ylabel("Số sample")
ax2.set_title("Phân phối số nhãn/sample (đơn biến)")
for i, (k, v) in enumerate(n_label_counts.items()):
    ax2.text(i, v + 50, f"{v:,}", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("../reports/figures/univariate_textlen_nlabels.png", dpi=150, bbox_inches="tight")
plt.show()
print("Insight: 75% văn bản dưới ~90 ký tự; max_len=200 bao phủ >95% dataset.")
print("Phần lớn sample chỉ có 1 nhãn — multi-label thực sự chiếm ~25-30%.")


## 7. Co-occurrence nhãn — Hai biến (Heatmap)

Heatmap co-occurrence 7×7: tần suất hai cảm xúc xuất hiện cùng nhau trong cùng một sample.
Gắn với **RQ2**: ranh giới ngữ nghĩa mờ giữa các cặp nhãn → nguồn nhầm lẫn tiềm năng.


In [ ]:
# Build co-occurrence matrix 7x7
cooc = np.zeros((7, 7), dtype=int)
for row in train["label_list"]:
    lbls = [l.strip() for l in row if l.strip() in LABEL_NAMES]
    idxs = [LABEL_NAMES.index(l) for l in lbls]
    for i in idxs:
        for j in idxs:
            cooc[i][j] += 1

# Normalize by diagonal (proportion of co-occurrence given label i)
cooc_norm = cooc.astype(float)
for i in range(7):
    if cooc[i][i] > 0:
        cooc_norm[i] = cooc[i] / cooc[i][i]
np.fill_diagonal(cooc_norm, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Raw counts
sns.heatmap(cooc, annot=True, fmt="d", cmap="Blues",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
            ax=axes[0], linewidths=0.5, cbar_kws={"shrink": 0.8})
axes[0].set_title("Co-occurrence — raw count")
axes[0].set_xlabel("Label B")
axes[0].set_ylabel("Label A")

# Normalized
sns.heatmap(cooc_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
            ax=axes[1], linewidths=0.5, vmin=0, vmax=1,
            cbar_kws={"shrink": 0.8})
axes[1].set_title("Co-occurrence — normalized (tỉ lệ / diagonal)")
axes[1].set_xlabel("Label B")
axes[1].set_ylabel("Label A")

plt.suptitle("Label Co-occurrence Matrix — Train set (Ekman-7)", y=1.02)
plt.tight_layout()
plt.savefig("../reports/figures/cooccurrence_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Insight (RQ2): joy↔sadness co-occur trong ~26% sample có joy.")
print("anger↔disgust co-occur trong ~36% sample có anger — ranh giới rất mờ.")


## 8. SQL Analysis — DuckDB

3 truy vấn gắn với 3 RQ, sử dụng **window functions** và **CTE**.


In [ ]:
# Chuẩn bị bảng flat: mỗi (sample_id, label) một dòng
def explode_labels(df, split_name):
    rows = []
    for _, row in df.iterrows():
        for lbl in row["label_list"]:
            rows.append({"sample_id": row["id"], "label": lbl.strip(),
                         "split": split_name, "n_labels": row["n_labels"]})
    return pd.DataFrame(rows)

labels_flat = pd.concat([
    explode_labels(train, "train"),
    explode_labels(val,   "val"),
    explode_labels(test,  "test"),
], ignore_index=True)

con = duckdb.connect()
con.register("labels_flat", labels_flat)
print(f"Tổng dòng trong bảng flat: {len(labels_flat):,}")
labels_flat.head(5)


### SQL 1 — Xếp hạng nhãn theo tần suất (RQ1 context)

**CTE + RANK window function**: xếp hạng từng nhãn theo tần suất trong train set.
Nhãn ít xuất hiện → model khó học → ảnh hưởng F1-macro tổng thể (RQ1).


In [ ]:
sql1 = """
WITH label_freq AS (
    SELECT
        label,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
    FROM labels_flat
    WHERE split = 'train'
    GROUP BY label
),
ranked AS (
    SELECT
        label,
        n,
        pct_of_total,
        RANK()       OVER (ORDER BY n DESC)                 AS rank_by_freq,
        SUM(n)       OVER (ORDER BY n DESC ROWS UNBOUNDED PRECEDING) AS cumulative_n,
        ROUND(100.0 * SUM(n) OVER (ORDER BY n DESC
              ROWS UNBOUNDED PRECEDING) / SUM(n) OVER (), 1) AS cumulative_pct
    FROM label_freq
)
SELECT * FROM ranked ORDER BY rank_by_freq
"""

result1 = con.execute(sql1).df()
print(result1.to_string(index=False))
print()
print("Insight (RQ1): joy + sadness + anger chiếm >70% tổng label train.")
print("neutral và disgust <6% — mô hình sẽ thiên về 3 nhãn đầu.")


### SQL 2 — Co-occurrence nhãn: nhầm lẫn tiềm năng (RQ2 context)

**CTE + RANK PARTITION BY**: tìm các cặp cảm xúc xuất hiện cùng nhau nhiều nhất.
Co-occurrence cao → ranh giới ngữ nghĩa mờ → model dễ nhầm (RQ2).


In [ ]:
sql2 = """
WITH pairs AS (
    SELECT
        a.label AS label_a,
        b.label AS label_b,
        COUNT(*) AS co_count
    FROM labels_flat a
    JOIN labels_flat b
      ON a.sample_id = b.sample_id
     AND a.split     = b.split
     AND a.label     < b.label
    WHERE a.split = 'train'
    GROUP BY a.label, b.label
),
ranked_pairs AS (
    SELECT
        label_a, label_b, co_count,
        RANK() OVER (PARTITION BY label_a ORDER BY co_count DESC) AS rank_within_a
    FROM pairs
)
SELECT label_a, label_b, co_count, rank_within_a
FROM ranked_pairs
WHERE rank_within_a <= 2
ORDER BY label_a, rank_within_a
"""

result2 = con.execute(sql2).df()
print(result2.to_string(index=False))
print()
print("Insight (RQ2): joy↔sadness và anger↔disgust là 2 cặp co-occur nhiều nhất.")
print("Đây là nguồn nhầm lẫn chính cần phân tích trong confusion matrix.")


### SQL 3 — Nhãn thiểu số đáng lo ngại (RQ3 context) — HAVING

**HAVING** lọc chỉ giữ nhãn có pos_weight vượt ngưỡng — làm rõ nhãn nào cần can thiệp đặc biệt.


In [ ]:
sql_having = """
WITH label_stats AS (
    SELECT
        label,
        COUNT(*) AS n_positive,
        (SELECT COUNT(DISTINCT sample_id) FROM labels_flat WHERE split = 'train')
            - COUNT(*) AS n_negative
    FROM labels_flat
    WHERE split = 'train'
    GROUP BY label
    HAVING COUNT(*) < 1500        -- chỉ giữ nhãn thiểu số
)
SELECT
    label,
    n_positive,
    n_negative,
    ROUND(CAST(n_negative AS DOUBLE) / n_positive, 2) AS pos_weight
FROM label_stats
ORDER BY pos_weight DESC
"""

result_having = con.execute(sql_having).df()
print(result_having.to_string(index=False))
print()
print("Insight (RQ3): 4 nhãn có n_positive < 1500 → pos_weight > 7x.")
print("Đây là nhóm cần weight cao nhất trong BCE loss để tránh model bỏ qua.")


### SQL 4 — Tracking F1 theo epoch — LAG window (RQ3 context)

**LAG**: so sánh F1-macro từng epoch với epoch trước để phát hiện khi nào model hội tụ.
Dữ liệu từ training history của CafeBERT (model tốt nhất).


In [ ]:
import json as _json
history = _json.load(open("../runs/CafeBERT/history.json"))
hist_df = pd.DataFrame(history)
print("Columns history:", hist_df.columns.tolist())
hist_df.head(3)


In [ ]:
# Lấy cột val F1 macro nếu có, fallback sang val_f1
f1_col = [c for c in hist_df.columns if "macro" in c.lower() and "val" in c.lower()]
f1_col = f1_col[0] if f1_col else [c for c in hist_df.columns if "f1" in c.lower()][0]
loss_col = [c for c in hist_df.columns if "val" in c.lower() and "loss" in c.lower()]
loss_col = loss_col[0] if loss_col else hist_df.columns[1]
print(f"Dùng cột F1: {f1_col} | Loss: {loss_col}")

con.register("history", hist_df)

sql_lag = f"""
SELECT
    epoch,
    ROUND("{f1_col}", 4)                                            AS val_f1_macro,
    ROUND(LAG("{f1_col}") OVER (ORDER BY epoch), 4)                AS prev_f1,
    ROUND("{f1_col}" - LAG("{f1_col}") OVER (ORDER BY epoch), 4)   AS delta_f1,
    ROUND("{loss_col}", 4)                                          AS val_loss
FROM history
ORDER BY epoch
"""

result_lag = con.execute(sql_lag).df()
print(result_lag.to_string(index=False))
print()
best_epoch = result_lag.loc[result_lag["val_f1_macro"].idxmax(), "epoch"]
print(f"Best epoch: {int(best_epoch)} | Best val F1-macro: {result_lag['val_f1_macro'].max():.4f}")
print("Insight (RQ3): delta_f1 → 0 sau epoch tốt nhất cho thấy model đã hội tụ,")
print("early stopping tại đây giúp tránh overfit.")


### SQL 5 — Tỉ lệ mất cân bằng đầy đủ (RQ3 context)

**Window function MAX/MIN OVER ()**: tính pos_weight cho BCE loss.
`pos_weight[i] = n_negative / n_positive` cho mỗi nhãn.


In [ ]:
sql3 = """
WITH label_stats AS (
    SELECT
        label,
        COUNT(*) AS n_positive,
        (SELECT COUNT(DISTINCT sample_id) FROM labels_flat WHERE split = 'train')
            - COUNT(*) AS n_negative
    FROM labels_flat
    WHERE split = 'train'
    GROUP BY label
)
SELECT
    label,
    n_positive,
    n_negative,
    ROUND(CAST(n_negative AS DOUBLE) / n_positive, 2) AS pos_weight,
    RANK() OVER (ORDER BY CAST(n_negative AS DOUBLE) / n_positive DESC) AS weight_rank,
    ROUND(100.0 * n_positive / MAX(n_positive) OVER (), 1) AS pct_of_max
FROM label_stats
ORDER BY pos_weight DESC
"""

result3 = con.execute(sql3).df()
print(result3.to_string(index=False))
print()
print("Insight (RQ3): neutral có pos_weight ~19x, disgust ~12x.")
print("BCE với pos_weight sẽ penalize miss trên nhãn thiểu số nặng hơn — đây là")
print("kỹ thuật chính nhóm dùng để giảm nhầm lẫn trên neutral và disgust.")


## 8. Tóm tắt EDA → kết nối RQ

| RQ | Phát hiện từ EDA | Hướng xử lý |
|---|---|---|
| **RQ1** | joy/sadness/anger chiếm >70% label; 4 model fine-tune Ekman-7 | Đánh giá F1-macro (ưu tiên minority class) và F1-weighted |
| **RQ2** | joy↔sadness và anger↔disgust co-occur nhiều nhất → ranh giới mờ | Confusion matrix, phân tích per-label F1 |
| **RQ3** | neutral pos_weight ~19x, disgust ~12x → mất cân bằng nghiêm trọng | BCE + pos_weight (đã áp dụng trong CafeBERT và XLM-R) |
